In [1]:
# CNN Avançada para Classificação COVID-QU-Ex Dataset (Normal, COVID-19, Non-COVID)
# Adaptado para o dataset COVID-QU-Ex do Kaggle

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import seaborn as sns
import pandas as pd
from collections import Counter
import cv2
from torchvision import models
import torch.nn.functional as F
import kagglehub
import shutil
from pathlib import Path

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

def download_and_prepare_dataset():
    """Baixa e prepara o dataset COVID-QU-Ex do Kaggle"""
    print("Baixando dataset COVID-QU-Ex...")
    
    # Download do dataset
    path = kagglehub.dataset_download("anasmohammedtahir/covidqu")
    print(f"Dataset baixado em: {path}")
    
    # Explorar estrutura do dataset
    print("\nExplorando estrutura do dataset...")
    for root, dirs, files in os.walk(path):
        level = root.replace(path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Mostrar apenas os primeiros 5 arquivos
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files) - 5} more files')
    
    return path

class COVIDQUDataset(Dataset):
    """Dataset personalizado para COVID-QU-Ex com pré-processamento avançado"""
    
    def __init__(self, dataset_path, split='train', transform=None, use_lung_masks=False):
        self.samples = []
        self.transform = transform
        self.use_lung_masks = use_lung_masks
        self.class_names = ['Normal', 'COVID-19', 'Non-COVID']
        
        # Mapeamento de labels
        self.label_mapping = {
            'Normal': 0,
            'COVID-19': 1, 
            'Non-COVID': 2
        }
        
        # Procurar pelos diretórios de dados
        self.dataset_path = Path(dataset_path)
        self._load_samples(split)
        
        print(f"Dataset {split} carregado: {len(self.samples)} amostras")
        self._print_class_distribution()
    
    def _find_dataset_structure(self):
        """Encontra a estrutura do dataset automaticamente"""
        possible_paths = [
            self.dataset_path / "Infection Segmentation Data" / "Infection Segmentation Data",
            self.dataset_path / "Lung Segmentation Data" / "Lung Segmentation Data", 
            self.dataset_path / "Infection Segmentation Data",
            self.dataset_path / "Lung Segmentation Data",
            self.dataset_path
        ]
        
        for path in possible_paths:
            if path.exists():
                # Verificar se tem estrutura de train/val/test
                train_path = path / "Train"
                if train_path.exists():
                    return path
                
                # Verificar estrutura alternativa
                for subdir in path.iterdir():
                    if subdir.is_dir() and any(x in subdir.name.lower() for x in ['train', 'test', 'val']):
                        return path
        
        return self.dataset_path
    
    def _load_samples(self, split):
        """Carrega amostras baseado no split especificado"""
        base_path = self._find_dataset_structure()
        
        # Tentar diferentes nomenclaturas de diretórios
        split_variants = {
            'train': ['Train', 'train', 'training'],
            'val': ['Val', 'val', 'validation', 'Validation'],
            'test': ['Test', 'test', 'testing']
        }
        
        split_path = None
        for variant in split_variants.get(split, [split]):
            potential_path = base_path / variant
            if potential_path.exists():
                split_path = potential_path
                break
        
        if split_path is None:
            # Se não encontrar estrutura train/val/test, tentar carregar tudo e fazer split manual
            print(f"Estrutura {split} não encontrada. Tentando carregar dataset completo...")
            self._load_all_and_split(base_path, split)
            return
        
        print(f"Carregando dados de: {split_path}")
        
        # Carregar imagens por classe
        for class_name in self.class_names:
            class_path = split_path / class_name
            if not class_path.exists():
                # Tentar variações do nome da classe
                class_variants = {
                    'Normal': ['Normal', 'normal', 'NORMAL'],
                    'COVID-19': ['COVID-19', 'covid', 'COVID', 'Covid-19'],
                    'Non-COVID': ['Non-COVID', 'non-covid', 'NonCOVID', 'Pneumonia']
                }
                
                for variant in class_variants.get(class_name, [class_name]):
                    variant_path = split_path / variant
                    if variant_path.exists():
                        class_path = variant_path
                        break
            
            if class_path.exists():
                label = self.label_mapping[class_name]
                for img_file in class_path.glob('*.png'):
                    self.samples.append((str(img_file), label))
                for img_file in class_path.glob('*.jpg'):
                    self.samples.append((str(img_file), label))
                for img_file in class_path.glob('*.jpeg'):
                    self.samples.append((str(img_file), label))
            else:
                print(f"Aviso: Classe {class_name} não encontrada em {split_path}")
    
    def _load_all_and_split(self, base_path, split):
        """Carrega todo o dataset e faz split manual"""
        all_samples = []
        
        # Procurar por todas as imagens
        for img_file in base_path.rglob('*.png'):
            # Tentar determinar a classe pelo caminho
            path_parts = img_file.parts
            class_name = None
            
            for part in path_parts:
                if 'normal' in part.lower():
                    class_name = 'Normal'
                    break
                elif 'covid' in part.lower() and 'non' not in part.lower():
                    class_name = 'COVID-19'
                    break
                elif 'non-covid' in part.lower() or 'pneumonia' in part.lower():
                    class_name = 'Non-COVID'
                    break
            
            if class_name and class_name in self.label_mapping:
                label = self.label_mapping[class_name]
                all_samples.append((str(img_file), label))
        
        # Fazer split manual
        if all_samples:
            from sklearn.model_selection import train_test_split
            
            # Separar por classe para split estratificado
            class_samples = {0: [], 1: [], 2: []}
            for sample, label in all_samples:
                class_samples[label].append((sample, label))
            
            # Split por classe
            train_samples, temp_samples = [], []
            val_samples, test_samples = [], []
            
            for label, samples in class_samples.items():
                if len(samples) > 0:
                    train, temp = train_test_split(samples, test_size=0.3, random_state=42)
                    val, test = train_test_split(temp, test_size=0.5, random_state=42)
                    
                    train_samples.extend(train)
                    val_samples.extend(val)
                    test_samples.extend(test)
            
            # Selecionar amostras baseado no split
            if split == 'train':
                self.samples = train_samples
            elif split == 'val':
                self.samples = val_samples
            elif split == 'test':
                self.samples = test_samples
    
    def _print_class_distribution(self):
        """Imprime a distribuição das classes"""
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        print("Distribuição das classes:")
        for class_idx, count in class_counts.items():
            class_name = self.class_names[class_idx]
            print(f"  {class_name}: {count} amostras")
    
    def get_class_weights(self):
        """Calcula pesos das classes para balanceamento"""
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        total_samples = len(self.samples)
        
        # Pesos inversamente proporcionais à frequência
        weights = []
        for i in range(len(self.class_names)):
            if i in class_counts:
                weight = total_samples / (len(self.class_names) * class_counts[i])
                weights.append(weight)
            else:
                weights.append(0.0)
        
        return torch.FloatTensor(weights)
    
    def apply_lung_mask(self, image, img_path):
        """Aplica máscara pulmonar se disponível"""
        try:
            # Procurar máscara correspondente
            mask_path = img_path.replace('.png', '_mask.png').replace('.jpg', '_mask.png')
            if not os.path.exists(mask_path):
                # Tentar outras convenções de nomenclatura
                base_name = os.path.splitext(os.path.basename(img_path))[0]
                mask_dir = os.path.dirname(img_path).replace('Images', 'Masks')
                mask_path = os.path.join(mask_dir, f"{base_name}_mask.png")
            
            if os.path.exists(mask_path):
                mask = Image.open(mask_path).convert('L')
                mask = mask.resize(image.size)
                
                # Aplicar máscara
                image_array = np.array(image)
                mask_array = np.array(mask)
                
                # Normalizar máscara
                mask_array = mask_array / 255.0
                
                # Aplicar máscara em cada canal
                if len(image_array.shape) == 3:
                    for c in range(image_array.shape[2]):
                        image_array[:, :, c] = image_array[:, :, c] * mask_array
                else:
                    image_array = image_array * mask_array
                
                return Image.fromarray(image_array.astype(np.uint8))
        except Exception as e:
            print(f"Erro ao aplicar máscara em {img_path}: {e}")
        
        return image
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
            
            # Aplicar máscara pulmonar se habilitada
            if self.use_lung_masks:
                image = self.apply_lung_mask(image, img_path)
            
            if self.transform:
                image = self.transform(image)
                
            return image, label
        except Exception as e:
            print(f"Erro ao carregar imagem {img_path}: {e}")
            # Retornar uma imagem preta em caso de erro
            dummy_image = torch.zeros(3, 224, 224)
            return dummy_image, label

# Transformações avançadas com Data Augmentation específicas para CXR
def get_transforms(phase='train'):
    """Define transformações específicas para cada fase"""
    
    if phase == 'train':
        # Data Augmentation conservador para imagens médicas
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),  # Rotação menor para CXR
            transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Ajustes menores
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        # Transformações para validação/teste
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])

class COVIDClassificationCNN(nn.Module):
    """CNN avançada para classificação COVID-19 com Transfer Learning"""
    
    def __init__(self, num_classes=3, pretrained=True, dropout_rate=0.4):
        super(COVIDClassificationCNN, self).__init__()
        
        # Backbone pré-treinado (ResNet50)
        self.backbone = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        
        # Substituir a última camada
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Classificador customizado otimizado para COVID-19
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
        
        # Inicialização das camadas customizadas
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Inicialização personalizada dos pesos"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

class ModelTrainer:
    """Classe para gerenciar o treinamento do modelo"""
    
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, 
                 scheduler=None, device='cpu'):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        
        # Histórico de treinamento
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        
    def train_epoch(self):
        """Treina por uma época"""
        self.model.train()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        for batch_idx, (images, labels) in enumerate(self.train_loader):
            images, labels = images.to(self.device), labels.to(self.device)
            
            # Forward pass
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            # Backward pass
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            # Estatísticas
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            
            # Log do progresso
            if batch_idx % 100 == 0:
                print(f'Batch {batch_idx}/{len(self.train_loader)}, '
                      f'Loss: {loss.item():.4f}')
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = correct_predictions / total_samples
        
        return epoch_loss, epoch_acc
    
    def validate_epoch(self):
        """Valida por uma época"""
        self.model.eval()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        
        with torch.no_grad():
            for images, labels in self.val_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct_predictions += (predicted == labels).sum().item()
                total_samples += labels.size(0)
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = correct_predictions / total_samples
        
        return epoch_loss, epoch_acc
    
    def train(self, num_epochs, early_stopping_patience=15):
        """Treinamento completo com early stopping"""
        best_val_acc = 0.0
        patience_counter = 0
        
        print(f"Iniciando treinamento por {num_epochs} épocas...")
        print("-" * 60)
        
        for epoch in range(num_epochs):
            print(f'Época {epoch+1}/{num_epochs}')
            
            # Treinamento
            train_loss, train_acc = self.train_epoch()
            
            # Validação
            val_loss, val_acc = self.validate_epoch()
            
            # Atualizar scheduler
            if self.scheduler:
                old_lr = self.optimizer.param_groups[0]['lr']
                self.scheduler.step(val_loss)
                new_lr = self.optimizer.param_groups[0]['lr']
                if new_lr != old_lr:
                    print(f'Learning rate reduzido de {old_lr:.6f} para {new_lr:.6f}')
            
            # Salvar histórico
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            
            print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
            print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
            
            # Early stopping
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                # Salvar melhor modelo
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_acc': val_acc,
                    'val_loss': val_loss
                }, 'best_covid_model.pth')
                print(f'Novo melhor modelo salvo! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                print(f'Early stopping após {early_stopping_patience} épocas sem melhoria')
                break
            
            print("-" * 60)
        
        print(f'Treinamento concluído! Melhor Val Acc: {best_val_acc:.4f}')
        return best_val_acc
    
    def plot_training_history(self):
        """Plota o histórico de treinamento"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Loss
        ax1.plot(self.train_losses, label='Train Loss', color='blue')
        ax1.plot(self.val_losses, label='Validation Loss', color='red')
        ax1.set_title('Curva de Loss - COVID-19 Classification')
        ax1.set_xlabel('Época')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)
        
        # Accuracy
        ax2.plot(self.train_accuracies, label='Train Accuracy', color='blue')
        ax2.plot(self.val_accuracies, label='Validation Accuracy', color='red')
        ax2.set_title('Curva de Acurácia - COVID-19 Classification')
        ax2.set_xlabel('Época')
        ax2.set_ylabel('Acurácia')
        ax2.legend()
        ax2.grid(True)
        
        plt.tight_layout()
        plt.savefig('covid_training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

def evaluate_model(model, test_loader, device, class_names):
    """Avaliação completa do modelo"""
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    # Métricas
    print("=== RELATÓRIO DE CLASSIFICAÇÃO COVID-19 ===")
    print(classification_report(all_labels, all_predictions, 
                              target_names=class_names, digits=4))
    
    # Matriz de confusão
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matriz de Confusão - COVID-19 Classification')
    plt.ylabel('Rótulo Verdadeiro')
    plt.xlabel('Rótulo Predito')
    plt.savefig('covid_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Calcular AUC para classificação multiclasse
    all_probabilities = np.array(all_probabilities)
    try:
        auc_scores = {}
        for i, class_name in enumerate(class_names):
            binary_labels = (np.array(all_labels) == i).astype(int)
            auc = roc_auc_score(binary_labels, all_probabilities[:, i])
            auc_scores[class_name] = auc
            print(f"AUC {class_name}: {auc:.4f}")
        
        mean_auc = np.mean(list(auc_scores.values()))
        print(f"AUC Médio: {mean_auc:.4f}")
        
        # Métricas específicas para COVID-19
        covid_precision = classification_report(all_labels, all_predictions, 
                                              target_names=class_names, 
                                              output_dict=True)['COVID-19']['precision']
        covid_recall = classification_report(all_labels, all_predictions, 
                                           target_names=class_names, 
                                           output_dict=True)['COVID-19']['recall']
        covid_f1 = classification_report(all_labels, all_predictions, 
                                       target_names=class_names, 
                                       output_dict=True)['COVID-19']['f1-score']
        
        print(f"\n=== MÉTRICAS ESPECÍFICAS COVID-19 ===")
        print(f"Precisão COVID-19: {covid_precision:.4f}")
        print(f"Recall COVID-19: {covid_recall:.4f}")
        print(f"F1-Score COVID-19: {covid_f1:.4f}")
        
    except Exception as e:
        print(f"Erro ao calcular AUC: {e}")
    
    return all_predictions, all_labels, all_probabilities

def main():
    """Função principal"""
    
    # Configurações otimizadas para COVID-QU-Ex
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.0001
    WEIGHT_DECAY = 1e-4
    USE_LUNG_MASKS = False  # Ativar se quiser usar máscaras pulmonares
    
    print("=== CNN AVANÇADA PARA CLASSIFICAÇÃO COVID-19 ===")
    print("Dataset: COVID-QU-Ex (Kaggle)")
    print("Classes: Normal, COVID-19, Non-COVID")
    print("=" * 60)
    
    try:
        # Baixar e preparar dataset
        dataset_path = download_and_prepare_dataset()
        
        # Carregar datasets
        print("\nCarregando datasets...")
        train_dataset = COVIDQUDataset(
            dataset_path, 
            split='train',
            transform=get_transforms('train'),
            use_lung_masks=USE_LUNG_MASKS
        )
        
        val_dataset = COVIDQUDataset(
            dataset_path, 
            split='val',
            transform=get_transforms('val'),
            use_lung_masks=USE_LUNG_MASKS
        )
        
        test_dataset = COVIDQUDataset(
            dataset_path, 
            split='test',
            transform=get_transforms('test'),
            use_lung_masks=USE_LUNG_MASKS
        )
        
        # Verificar se temos dados suficientes
        if len(train_dataset) == 0:
            print("ERRO: Nenhuma amostra de treinamento encontrada!")
            return
        
        # Calcular pesos das classes
        class_weights = train_dataset.get_class_weights().to(device)
        print(f"\nPesos das classes: {class_weights}")
        
        # DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, 
                                shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                              shuffle=False, num_workers=4, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, 
                               shuffle=False, num_workers=4, pin_memory=True)
        
        # Modelo
        print("\nInicializando modelo COVID-19...")
        model = COVIDClassificationCNN(num_classes=3, pretrained=True, dropout_rate=0.4)
        model = model.to(device)
        
        # Critério com pesos das classes
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        # Otimizador
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, 
                              weight_decay=WEIGHT_DECAY)
        
        # Scheduler
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, verbose=True
        )
        
        # Treinador
        trainer = ModelTrainer(model, train_loader, val_loader, criterion, 
                             optimizer, scheduler, device)
        
        # Treinamento
        best_val_acc = trainer.train(NUM_EPOCHS, early_stopping_patience=15)
        
        # Plotar histórico
        trainer.plot_training_history()
        
        # Carregar melhor modelo para avaliação
        if os.path.exists('best_covid_model.pth'):
            print("\nCarregando melhor modelo para avaliação final...")
            checkpoint = torch.load('best_covid_model.pth', map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
        
        # Avaliação final
        print("\n=== AVALIAÇÃO NO CONJUNTO DE TESTE ===")
        class_names = ['Normal', 'COVID-19', 'Non-COVID']
        predictions, labels, probabilities = evaluate_model(
            model, test_loader, device, class_names
        )
        
        print("\n=== RESUMO FINAL ===")
        print(f"Melhor acurácia de validação: {best_val_acc:.4f}")
        print(f"Total de amostras de treinamento: {len(train_dataset)}")
        print(f"Total de amostras de validação: {len(val_dataset)}")
        print(f"Total de amostras de teste: {len(test_dataset)}")
        
        # Salvar estatísticas
        stats = {
            'best_val_acc': best_val_acc,
            'train_samples': len(train_dataset),
            'val_samples': len(val_dataset),
            'test_samples': len(test_dataset),
            'class_weights': class_weights.cpu().numpy().tolist()
        }
        
        import json
        with open('covid_training_stats.json', 'w') as f:
            json.dump(stats, f, indent=2)
        print("\nTreinamento concluído com sucesso!")
        
    except Exception as e:
        print(f"Erro durante a execução: {e}")
        import traceback
        traceback.print_exc()

def predict_single_image(model, image_path, transform, device, class_names):
    """Predição em uma única imagem"""
    model.eval()
    
    # Carregar e preprocessar imagem
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
    
    predicted_class = class_names[predicted.item()]
    confidence = probabilities[0][predicted.item()].item()
    
    print(f"Imagem: {image_path}")
    print(f"Predição: {predicted_class}")
    print(f"Confiança: {confidence:.4f}")
    
    # Mostrar probabilidades para todas as classes
    print("Probabilidades por classe:")
    for i, class_name in enumerate(class_names):
        prob = probabilities[0][i].item()
        print(f"  {class_name}: {prob:.4f}")
    
    return predicted_class, confidence

def visualize_predictions(model, test_loader, device, class_names, num_samples=16):
    """Visualiza predições em um conjunto de amostras"""
    model.eval()
    
    # Coletar amostras
    images_to_show = []
    labels_to_show = []
    predictions_to_show = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            if len(images_to_show) >= num_samples:
                break
                
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            # Desnormalizar imagens para visualização
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            
            for i in range(min(images.size(0), num_samples - len(images_to_show))):
                img = images[i].cpu() * std + mean
                img = torch.clamp(img, 0, 1)
                
                images_to_show.append(img.permute(1, 2, 0).numpy())
                labels_to_show.append(labels[i].item())
                predictions_to_show.append(predicted[i].item())
    
    # Plotar amostras
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    axes = axes.ravel()
    
    for i in range(min(len(images_to_show), 16)):
        ax = axes[i]
        ax.imshow(images_to_show[i], cmap='gray')
        
        true_label = class_names[labels_to_show[i]]
        pred_label = class_names[predictions_to_show[i]]
        
        color = 'green' if labels_to_show[i] == predictions_to_show[i] else 'red'
        ax.set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)
        ax.axis('off')
    
    # Ocultar eixos vazios
    for i in range(len(images_to_show), 16):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig('covid_predictions_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()

def analyze_misclassifications(predictions, labels, class_names):
    """Analisa erros de classificação"""
    print("\n=== ANÁLISE DE ERROS DE CLASSIFICAÇÃO ===")
    
    # Matriz de confusão detalhada
    cm = confusion_matrix(labels, predictions)
    
    print("\nErros mais comuns:")
    for i, true_class in enumerate(class_names):
        for j, pred_class in enumerate(class_names):
            if i != j and cm[i][j] > 0:
                error_rate = cm[i][j] / np.sum(cm[i])
                print(f"{true_class} classificado como {pred_class}: "
                      f"{cm[i][j]} casos ({error_rate:.2%})")
    
    # Análise específica para COVID-19
    covid_idx = class_names.index('COVID-19')
    covid_true_positives = cm[covid_idx][covid_idx]
    covid_false_negatives = np.sum(cm[covid_idx]) - covid_true_positives
    covid_false_positives = np.sum(cm[:, covid_idx]) - covid_true_positives
    
    print(f"\n=== ANÁLISE ESPECÍFICA COVID-19 ===")
    print(f"Verdadeiros Positivos: {covid_true_positives}")
    print(f"Falsos Negativos: {covid_false_negatives}")
    print(f"Falsos Positivos: {covid_false_positives}")
    
    if covid_false_negatives > 0:
        print(f"Taxa de Falsos Negativos: {covid_false_negatives/(covid_true_positives + covid_false_negatives):.2%}")
    if covid_false_positives > 0:
        normal_idx = class_names.index('Normal')
        non_covid_idx = class_names.index('Non-COVID')
        fn_from_normal = cm[normal_idx][covid_idx]
        fn_from_non_covid = cm[non_covid_idx][covid_idx]
        print(f"Falsos Positivos - Normal como COVID: {fn_from_normal}")
        print(f"Falsos Positivos - Non-COVID como COVID: {fn_from_non_covid}")

def create_covid_report(model, test_loader, device, class_names, dataset_stats):
    """Cria relatório detalhado do modelo COVID-19"""
    print("\n" + "="*80)
    print("RELATÓRIO DETALHADO - CLASSIFICADOR COVID-19 CNN")
    print("Dataset: COVID-QU-Ex")
    print("="*80)
    
    # Informações do dataset
    print(f"\nINFORMAÇÕES DO DATASET:")
    print(f"Total de imagens: 33,920")
    print(f"COVID-19: 11,956 imagens")
    print(f"Non-COVID: 11,263 imagens") 
    print(f"Normal: 10,701 imagens")
    print(f"Amostras de treino utilizadas: {dataset_stats.get('train_samples', 'N/A')}")
    print(f"Amostras de validação: {dataset_stats.get('val_samples', 'N/A')}")
    print(f"Amostras de teste: {dataset_stats.get('test_samples', 'N/A')}")
    
    # Avaliação detalhada
    predictions, labels, probabilities = evaluate_model(model, test_loader, device, class_names)
    
    # Análise de erros
    analyze_misclassifications(predictions, labels, class_names)
    
    # Visualizar predições
    visualize_predictions(model, test_loader, device, class_names)
    
    # Métricas por classe
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, support = precision_recall_fscore_support(labels, predictions)
    
    print(f"\n=== MÉTRICAS DETALHADAS POR CLASSE ===")
    for i, class_name in enumerate(class_names):
        print(f"\n{class_name.upper()}:")
        print(f"  Precisão: {precision[i]:.4f}")
        print(f"  Recall: {recall[i]:.4f}")
        print(f"  F1-Score: {f1[i]:.4f}")
        print(f"  Suporte: {support[i]} amostras")
    
    # Salvar relatório em arquivo
    with open('covid_classification_report.txt', 'w', encoding='utf-8') as f:
        f.write("RELATÓRIO DETALHADO - CLASSIFICADOR COVID-19 CNN\n")
        f.write("Dataset: COVID-QU-Ex\n")
        f.write("="*80 + "\n\n")
        
        # Adicionar métricas principais
        from sklearn.metrics import accuracy_score
        accuracy = accuracy_score(labels, predictions)
        f.write(f"Acurácia Geral: {accuracy:.4f}\n\n")
        
        # Relatório de classificação
        report = classification_report(labels, predictions, target_names=class_names, digits=4)
        f.write("RELATÓRIO DE CLASSIFICAÇÃO:\n")
        f.write(report)
        f.write("\n")
        
        # Matriz de confusão
        cm = confusion_matrix(labels, predictions)
        f.write("MATRIZ DE CONFUSÃO:\n")
        f.write("Formato: [Normal, COVID-19, Non-COVID]\n")
        for i, row in enumerate(cm):
            f.write(f"{class_names[i]}: {row}\n")
    
    print(f"\nRelatório salvo em: covid_classification_report.txt")
    return predictions, labels, probabilities

def save_model_for_deployment(model, transform_params, class_names, model_info):
    """Salva modelo para deployment"""
    deployment_package = {
        'model_state_dict': model.state_dict(),
        'model_architecture': 'ResNet50-based COVID Classifier',
        'class_names': class_names,
        'transform_params': {
            'mean': [0.485, 0.456, 0.406],
            'std': [0.229, 0.224, 0.225],
            'size': (224, 224)
        },
        'model_info': model_info,
        'dataset': 'COVID-QU-Ex',
        'num_classes': len(class_names)
    }
    
    torch.save(deployment_package, 'covid_classifier_deployment.pth')
    print("Modelo salvo para deployment: covid_classifier_deployment.pth")

# Atualizar função main() com melhorias
if __name__ == "__main__":
    try:
        main()
        
        # Após o treinamento principal, executar análises adicionais
        if os.path.exists('best_covid_model.pth'):
            print("\n" + "="*60)
            print("EXECUTANDO ANÁLISES COMPLEMENTARES...")
            print("="*60)
            
            # Recarregar dados para análise final
            dataset_path = kagglehub.dataset_download("anasmohammedtahir/covidqu")
            
            test_dataset = COVIDQUDataset(
                dataset_path, 
                split='test',
                transform=get_transforms('test'),
                use_lung_masks=False
            )
            
            test_loader = DataLoader(test_dataset, batch_size=32, 
                                   shuffle=False, num_workers=4, pin_memory=True)
            
            # Carregar melhor modelo
            model = COVIDClassificationCNN(num_classes=3, pretrained=True, dropout_rate=0.4)
            model = model.to(device)
            
            checkpoint = torch.load('best_covid_model.pth', map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            
            # Estatísticas do dataset
            dataset_stats = {
                'train_samples': len(COVIDQUDataset(dataset_path, 'train', transform=None)),
                'val_samples': len(COVIDQUDataset(dataset_path, 'val', transform=None)),
                'test_samples': len(test_dataset),
                'best_val_acc': checkpoint.get('val_acc', 0.0)
            }
            
            # Relatório completo
            class_names = ['Normal', 'COVID-19', 'Non-COVID']
            predictions, labels, probabilities = create_covid_report(
                model, test_loader, device, class_names, dataset_stats
            )
            
            # Salvar modelo para deployment
            model_info = {
                'training_epochs': checkpoint.get('epoch', 0),
                'best_val_accuracy': checkpoint.get('val_acc', 0.0),
                'best_val_loss': checkpoint.get('val_loss', 0.0),
                'architecture': 'ResNet50 + Custom Classifier',
                'dropout_rate': 0.4,
                'optimizer': 'AdamW',
                'dataset_size': sum(dataset_stats.values())
            }
            
            save_model_for_deployment(model, None, class_names, model_info)
            
            print("\n" + "="*60)
            print("ANÁLISES CONCLUÍDAS!")
            print("Arquivos gerados:")
            print("- best_covid_model.pth (melhor modelo)")
            print("- covid_classifier_deployment.pth (modelo para deployment)")
            print("- covid_training_history.png (curvas de treinamento)")
            print("- covid_confusion_matrix.png (matriz de confusão)")
            print("- covid_predictions_visualization.png (visualização)")
            print("- covid_classification_report.txt (relatório detalhado)")
            print("- covid_training_stats.json (estatísticas)")
            print("="*60)
            
    except KeyboardInterrupt:
        print("\nTreinamento interrompido pelo usuário.")
    except Exception as e:
        print(f"Erro durante a execução: {e}")
        import traceback
        traceback.print_exc()

Dispositivo utilizado: cuda
=== CNN AVANÇADA PARA CLASSIFICAÇÃO COVID-19 ===
Dataset: COVID-QU-Ex (Kaggle)
Classes: Normal, COVID-19, Non-COVID
Baixando dataset COVID-QU-Ex...


100%|██████████| 1.15G/1.15G [00:41<00:00, 29.9MB/s]

Extracting files...


Dataset baixado em: /home/jose/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7

Explorando estrutura do dataset...
7/
  COVID-QU-Ex dataset.txt
  Infection Segmentation Data/
    Infection Segmentation Data/
      Val/
        Normal/
          infection masks/
            Normal (672).png
            Normal (10255).png
            Normal (673).png
            Normal (647).png
            Normal (10275).png
            ... and 228 more files
          lung masks/
            Normal (672).png
            Normal (10255).png
            Normal (673).png
            Normal (647).png
            Normal (10275).png
            ... and 228 more files
          images/
            Normal (672).png
            Normal (10255).png
            Normal (673).png
            Normal (647).png
            Normal (10275).png
            ... and 228 more files
        Non-COVID/
          infection masks/
            non_COVID (4912).png
            non_COVID (4925).png
            non_COV